In [36]:
import struct
import numpy as np


"""Попытка прочитать базовые поля из сериализованного DefectResults"""
with open('./thicks_export/thick_378451.raw', 'rb') as f:
    # Пропуск заголовка сериализации (~20-50 байт)
    # Это упрощённый пример — реальная структура сложнее
    try:
        x = struct.unpack('>i', f.read(4))[0]      # big-endian int
        y = struct.unpack('>i', f.read(4))[0]
        
       
        print(f"Размеры: {x}×{y}")
        #return doubles
    except Exception as e:
        print(f"⚠️ Не удалось распарсить: {e}")
#return None

Размеры: 188×8


In [38]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import struct
import os

def read_raw_file(filepath):
    """Чтение бинарного файла, созданного Java DataOutputStream (новый формат)"""
    with open(filepath, 'rb') as f:
        # Чтение заголовка (Java использует big-endian)
        x = struct.unpack('>i', f.read(4))[0]           # counts.length (X)
        y = struct.unpack('>i', f.read(4))[0]           # counts[0].length (Y)
        
        print(f"📊 Заголовок: {x}×{y}")
        
        # Чтение данных: double (8 байт, big-endian)
        # Данные представляют собой усреднённые значения [X][Y]
        total_elements = x * y
        data_flat = np.frombuffer(f.read(8 * total_elements), dtype='>f8')
        
        # Формирование 2D массива [X][Y]
        data = data_flat.reshape((x, y))
        
        return {
            'x': x, 'y': y,
            'data': data
        }


def create_defect_cmap(threshold=1.0):
    """Создание цветовой карты: значения < порога → зелёный, ≥ порога → красный"""
    # Создаём градиент из 256 цветов
    colors = []
    
    # Зелёная часть (норма): от светло-зелёного к насыщенному
    for i in range(128):
        intensity = 0.3 + 0.7 * (i / 127)
        colors.append((0, intensity, 0))  # (R, G, B)
    
    # Жёлтая часть (переходная зона)
    for i in range(64):
        colors.append((i/64, 1.0, 0))
    
    # Красная часть (дефект): от оранжевого к тёмно-красному
    for i in range(64):
        colors.append((1.0, 1.0 - i/64, 0))
    
    return mcolors.ListedColormap(colors)


def visualize_heatmap(data_2d, output_path=None, threshold=1.0, 
                      x_size=None, y_size=None, title="Толщинометрия"):
    """
    Визуализация 2D карты значений.
    data_2d: массив [X][Y] с усреднёнными значениями
    """
    x, y = data_2d.shape
    
    # Создаём цветовую карту
    cmap = create_defect_cmap(threshold)
    
    # Нормализация: центрируем вокруг порога
    vmin = min(threshold * 0.5, np.min(data_2d))
    vmax = max(threshold * 1.5, np.max(data_2d))
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=threshold, vmax=vmax)
    
    # Построение изображения
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # transpose для корректной ориентации (Y по вертикали)
    # origin='lower' чтобы (0,0) было в левом нижнем углу
    im = ax.imshow(data_2d.T, cmap=cmap, norm=norm,
                   origin='lower', interpolation='bilinear',
                   extent=[0, x_size or x, 0, y_size or y])
    
    # Добавляем линию порога на цветовой шкале
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Нормированное значение', rotation=270, labelpad=20)
    
    # Добавляем маркер порога на colorbar
    cbar.ax.axhline(y=(threshold - vmin) / (vmax - vmin), 
                    color='black', linestyle='--', linewidth=1)
    cbar.ax.text(1.02, (threshold - vmin) / (vmax - vmin), 
                 f'Порог: {threshold}', va='center', fontsize=9)
    
    plt.title(f'{title}\n(цвет: 🟢 норма < {threshold} ≤ 🔴 дефект)')
    plt.xlabel('X' + (f' (мм: 0–{x_size})' if x_size else ' (ячейки)'))
    plt.ylabel('Y' + (f' (мм: 0–{y_size})' if y_size else ' (ячейки)'))
    
    # Добавляем сетку
    ax.grid(True, linestyle=':', alpha=0.3)
    
    # Статистика на графике
    stats_text = (f"Среднее: {np.mean(data_2d):.3f}\n"
                  f"Мин: {np.min(data_2d):.3f}\n"
                  f"Макс: {np.max(data_2d):.3f}\n"
                  f"⚠ >порога: {np.sum(data_2d >= threshold)}/{x*y} "
                  f"({100*np.sum(data_2d >= threshold)/(x*y):.1f}%)")
    
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=props)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"💾 Сохранено: {output_path}")
    else:
        plt.show()
    
    plt.close()
    return fig


def process_directory(input_dir, output_dir='./output', threshold=1.0):
    """Обработка всех .raw файлов в директории"""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    raw_files = list(Path(input_dir).glob('thick_*.raw'))
    print(f"🔍 Найдено файлов: {len(raw_files)}")
    
    results = []
    
    for filepath in raw_files:
        print(f"\n📁 Обработка: {filepath.name}")
        try:
            # Чтение данных
            raw_data = read_raw_file(str(filepath))
            data = raw_data['data']
            
            # Визуализация
            output_path = os.path.join(output_dir, f'{filepath.stem}.png')
            visualize_heatmap(data, output_path, threshold=threshold,
                            title=f"ID: {filepath.stem.split('_')[-1]}")
            
            # Сбор статистики
            stats = {
                'file': filepath.name,
                'shape': data.shape,
                'mean': float(np.mean(data)),
                'std': float(np.std(data)),
                'min': float(np.min(data)),
                'max': float(np.max(data)),
                'defect_pct': float(100 * np.sum(data >= threshold) / data.size)
            }
            results.append(stats)
            print(f"✅ {filepath.name}: ⚠ дефекты {stats['defect_pct']:.1f}%")
            
        except Exception as e:
            print(f"❌ Ошибка обработки {filepath.name}: {e}")
    
    # Сводная статистика
    if results:
        print("\n📊 Сводная статистика:")
        print(f"{'Файл':<25} {'Среднее':>8} {'Стд.откл.':>10} {'Дефекты':>8}")
        print("-" * 55)
        for r in results:
            print(f"{r['file']:<25} {r['mean']:8.3f} {r['std']:10.3f} {r['defect_pct']:7.1f}%")
    
    return results


# 🔧 Пример использования
if __name__ == '__main__':
    # Настройки
    INPUT_DIR = './thicks_export'      # Директория с .raw файлами
    OUTPUT_DIR = './visualization_results'  # Директория для результатов
    THRESHOLD = 5.0                     # Порог дефекта (как в Java-коде)
    
    # Обработка одного файла (для теста)
    test_file = './thicks_export/thick_378451.raw'
    if os.path.exists(test_file):
        print(f"🧪 Тестовый режим: {test_file}")
        raw_data = read_raw_file(test_file)
        data = raw_data['data']
        
        print(f"📐 Размер матрицы: {data.shape}")
        print(f"📈 Диапазон значений: [{np.min(data):.3f}, {np.max(data):.3f}]")
        print(f"📊 Среднее: {np.mean(data):.3f} ± {np.std(data):.3f}")
        print(f"⚠ Ячеек выше порога ({THRESHOLD}): "
              f"{np.sum(data >= THRESHOLD)}/{data.size} "
              f"({100*np.sum(data >= THRESHOLD)/data.size:.1f}%)")
        
        # Визуализация
        visualize_heatmap(data, output_path='./test_output.png', 
                         threshold=THRESHOLD, title='Тестовый образец')
    
    # Массовая обработка (раскомментируйте для использования)
    # process_directory(INPUT_DIR, OUTPUT_DIR, threshold=THRESHOLD)

🧪 Тестовый режим: ./thicks_export/thick_378451.raw
📊 Заголовок: 188×8
📐 Размер матрицы: (188, 8)
📈 Диапазон значений: [0.000, 5.650]
📊 Среднее: 0.302 ± 1.193
⚠ Ячеек выше порога (5.0): 53/1504 (3.5%)


C:\Users\strel\AppData\Local\Temp\ipykernel_1256\2312365076.py:105: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\strel\AppData\Local\Temp\ipykernel_1256\2312365076.py:105: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\strel\AppData\Local\Temp\ipykernel_1256\2312365076.py:108: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=150, bbox_inches='tight')
C:\Users\strel\AppData\Local\Temp\ipykernel_1256\2312365076.py:108: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=150, bbox_inches='tight')


💾 Сохранено: ./test_output.png
